In [1]:
import pandas as pd
import requests
import datetime
import time
from tqdm import tqdm

print("Libraries imported.")

Libraries imported.


In [2]:
# Load the Hornets game data
file_path = 'data/hornets_games_2015-2026.csv'
df = pd.read_csv(file_path)

# Convert date column to datetime
df['date'] = pd.to_datetime(df['date'], format='%m/%d/%Y')

print(f"Loaded {len(df)} games.")
df.head()

Loaded 875 games.


,game_id,date,opponent,city,state,arena,attendance,capacity,attendance_percent,hornets_wins,...,hornets_score,opponent_score,location,hornets_top_scorer (points),opponents_top_scorer,hornets_most_assists,opponents_most_assists,hornets_top_rebounder,opponents_top_rebounder,season
0,400578776,2015-01-03,Cleveland Cavaliers,Charlotte,NC,Spectrum Center,19307,19077.0,101.2,10,...,87,91,Home,Gerald Henderson (14),Kevin Love (27),Kemba Walker (5),Matthew Dellavedova (4),Michael Kidd-Gilchrist (10),Tristan Thompson (14),2015
1,400579337,2015-03-22,Minnesota Timberwolves,Minneapolis,MN,Target Center,15262,18024.0,84.7,30,...,109,98,Away,Mo Williams (24),Gorgui Dieng (16),Kemba Walker (8),Kevin Martin (9),Al Jefferson (11),Adreian Payne (9),2015
2,400578701,2014-12-23,Denver Nuggets,Charlotte,NC,Spectrum Center,16913,19077.0,88.7,9,...,110,82,Home,Al Jefferson (22),Ty Lawson (18),Kemba Walker (9),Ty Lawson (4),P.J. Hairston (10),Kenneth Faried (9),2015
3,400578349,2014-11-06,Miami Heat,Charlotte,NC,Spectrum Center,15874,19077.0,83.2,2,...,96,89,Home,Al Jefferson (28),Dwyane Wade (25),Kemba Walker (7),Dwyane Wade (7),Lance Stephenson (13),Chris Bosh (13),2015
4,400579352,2015-03-25,Brooklyn Nets,Charlotte,NC,Spectrum Center,15091,19077.0,79.1,30,...,88,91,Home,Al Jefferson (23),Brook Lopez (34),Kemba Walker (6),Deron Williams (14),Al Jefferson (10),Brook Lopez (10),2015


In [3]:
# Filter for Home games 
# Check if 'location' column is 'Home' 
home_games = df[df['location'] == 'Home'].copy()

# Ensure we have a list of unique dates
home_dates = home_games['date'].dt.strftime('%Y-%m-%d').unique()
print(f"Found {len(home_games)} home games.")
print(f"Date range: {home_dates.min()} to {home_dates.max()}")

home_games.head()

Found 436 home games.
Date range: 2014-10-29 to 2025-04-08


,game_id,date,opponent,city,state,arena,attendance,capacity,attendance_percent,hornets_wins,...,hornets_score,opponent_score,location,hornets_top_scorer (points),opponents_top_scorer,hornets_most_assists,opponents_most_assists,hornets_top_rebounder,opponents_top_rebounder,season
0,400578776,2015-01-03,Cleveland Cavaliers,Charlotte,NC,Spectrum Center,19307,19077.0,101.2,10,...,87,91,Home,Gerald Henderson (14),Kevin Love (27),Kemba Walker (5),Matthew Dellavedova (4),Michael Kidd-Gilchrist (10),Tristan Thompson (14),2015
2,400578701,2014-12-23,Denver Nuggets,Charlotte,NC,Spectrum Center,16913,19077.0,88.7,9,...,110,82,Home,Al Jefferson (22),Ty Lawson (18),Kemba Walker (9),Ty Lawson (4),P.J. Hairston (10),Kenneth Faried (9),2015
3,400578349,2014-11-06,Miami Heat,Charlotte,NC,Spectrum Center,15874,19077.0,83.2,2,...,96,89,Home,Al Jefferson (28),Dwyane Wade (25),Kemba Walker (7),Dwyane Wade (7),Lance Stephenson (13),Chris Bosh (13),2015
4,400579352,2015-03-25,Brooklyn Nets,Charlotte,NC,Spectrum Center,15091,19077.0,79.1,30,...,88,91,Home,Al Jefferson (23),Brook Lopez (34),Kemba Walker (6),Deron Williams (14),Al Jefferson (10),Brook Lopez (10),2015
6,400579182,2015-03-04,Los Angeles Lakers,Charlotte,NC,Spectrum Center,16947,19077.0,88.8,25,...,104,103,Home,Al Jefferson (21),Jeremy Lin (23),Mo Williams (13),Jeremy Lin (8),Al Jefferson (16),Carlos Boozer (11),2015


In [4]:
# Define Charlotte, NC Coordinates
LAT = 35.2271
LON = -80.8431

# Open-Meteo Historical Weather API
# We can request all data in one go by specifying the start and end date
# This is much faster than looping per day
start_date = home_dates.min()
end_date = home_dates.max()

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": LAT,
    "longitude": LON,
    "start_date": start_date,
    "end_date": end_date,
    "daily": ["temperature_2m_max", "temperature_2m_min", "precipitation_sum", "rain_sum", "snowfall_sum", "weather_code", "wind_speed_10m_max"],
    "timezone": "America/New_York",
    "temperature_unit": "fahrenheit",
    "wind_speed_unit": "mph",
    "precipitation_unit": "inch"
}

print(f"Fetching weather data from {start_date} to {end_date}...")
response = requests.get(url, params=params)

if response.status_code == 200:
    weather_json = response.json()
    daily_data = weather_json['daily']
    
    # Create a DataFrame from the daily data
    weather_df = pd.DataFrame(daily_data)
    print("Weather data fetched successfully.")
else:
    print(f"Error fetching weather data: {response.status_code}")
    
weather_df.head()

Fetching weather data from 2014-10-29 to 2025-04-08...
Weather data fetched successfully.


,time,temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,snowfall_sum,weather_code,wind_speed_10m_max
0,2014-10-29,77.8,59.1,0.035,0.035,0.0,51,9.6
1,2014-10-30,66.5,51.4,0.000,0.000,0.0,3,7.9
2,2014-10-31,63.8,44.4,0.126,0.126,0.0,53,5.7
3,2014-11-01,50.6,41.4,0.720,0.720,0.0,61,15.5
4,2014-11-02,52.8,36.9,0.000,0.000,0.0,0,13.5


In [ ]:
# Function to interpret WMO Weather Codes
# Source: https://open-meteo.com/en/docs
def interpret_weather_code(code):
    if code == 0: return "Clear sky"
    if code in [1, 2, 3]: return "Partly cloudy"
    if code in [45, 48]: return "Fog"
    if code in [51, 53, 55]: return "Drizzle"
    if code in [56, 57]: return "Freezing Drizzle"
    if code in [61, 63, 65]: return "Rain"
    if code in [66, 67]: return "Freezing Rain"
    if code in [71, 73, 75]: return "Snow"
    if code == 77: return "Snow Flurries"
    if code in [80, 81, 82]: return "Rain"
    if code in [85, 86]: return "Snow"
    if code == 95: return "Thunderstorm"
    if code in [96, 99]: return "Thunderstorm with hail"
    return "Unknown"

# Format weather dataframe
weather_df['date'] = pd.to_datetime(weather_df['time'])
weather_df['weather_description'] = weather_df['weather_code'].apply(interpret_weather_code)

# Rename columns for clarity
weather_df = weather_df.rename(columns={
    'temperature_2m_max': 'temp_high',
    'temperature_2m_min': 'temp_low',
    'precipitation_sum': 'precipitation',
    'wind_speed_10m_max': 'wind_speed'
})

# Add binary variable for windy (20-30 mph)
weather_df['windy'] = ((weather_df['wind_speed'] >= 20)).astype(int)

# Keep relevant columns
weather_cols = ['date', 'temp_high', 'temp_low', 'precipitation', 'wind_speed', 'windy', 'weather_description']
weather_clean = weather_df[weather_cols].copy()

print("Weather data processed.")
weather_clean.head()

Weather data processed.


,date,temp_high,temp_low,precipitation,wind_speed,windy,weather_description
0,2014-10-29,77.8,59.1,0.035,9.6,0,Drizzle
1,2014-10-30,66.5,51.4,0.000,7.9,0,Partly cloudy
2,2014-10-31,63.8,44.4,0.126,5.7,0,Drizzle
3,2014-11-01,50.6,41.4,0.720,15.5,0,Rain
4,2014-11-02,52.8,36.9,0.000,13.5,0,Clear sky


In [9]:
# Merge weather data with home games to include game_id
# We want just the weather data for these specific dates, plus the game_id
weather_home_games_only = pd.merge(home_games[['date', 'game_id']], weather_clean, on='date', how='inner')

print(f"Weather rows matching home games: {len(weather_home_games_only)}")
pd.set_option('display.max_columns', None)
weather_home_games_only.head()

Weather rows matching home games: 436


,date,game_id,temp_high,temp_low,precipitation,wind_speed,windy,weather_description
0,2015-01-03,400578776,50.7,42.5,0.272,5.7,0,Rain
1,2014-12-23,400578701,44.3,37.2,0.437,7.3,0,Rain
2,2014-11-06,400578349,66.7,53.1,0.083,14.4,0,Drizzle
3,2015-03-25,400579352,68.2,50.5,0.028,8.5,0,Drizzle
4,2015-03-04,400579182,70.2,44.0,0.008,17.9,0,Drizzle


In [7]:
sorted(df['weather_description'].dropna().unique())

['Clear sky', 'Drizzle', 'Partly cloudy', 'Rain', 'Snow fall']

In [10]:
# Save weather data to CSV (overwriting previous file or creating new one)
output_file = 'data/weather_data.csv'
weather_home_games_only.to_csv(output_file, index=False)
print(f"Saved weather data for home games to {output_file}")

Saved weather data for home games to data/weather_data.csv
